## Cross Validation Implementation

In [1]:
# importing the libraries
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

In [2]:
# data
data = load_breast_cancer()

X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (569, 30)
y shape: (569,)


In [3]:
# k-fold cross validation
def k_fold_cross_validation(model, X, y, k=5, shuffle=True, random_state=42):
    n_samples = len(X)

    indices = np.arange(n_samples)

    if shuffle:
        rng = np.random.default_rng(random_state)
        rng.shuffle(indices)

    folds = np.array_split(indices, k)

    scores = []

    for i in range(k):
        validation_indices = folds[i]

        training_indices = np.concatenate(
            [folds[j] for j in range(k) if j != i]
        )

        X_train = X[training_indices]
        y_train = y[training_indices]

        X_validation = X[validation_indices]
        y_validation = y[validation_indices]

        model.fit(X_train, y_train)

        score = model.score(X_validation, y_validation)
        scores.append(score)

    return np.array(scores)

In [4]:
# model
model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

k = 5

scratch_scores = k_fold_cross_validation(
    model,
    X,
    y,
    k=k,
    shuffle=True,
    random_state=42
)

print("Fold-by-fold scores:")
for i, score in enumerate(scratch_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean: {scratch_scores.mean():.4f}")
print(f"Std:  {scratch_scores.std():.4f}")

Fold-by-fold scores:
Fold 1: 0.9474
Fold 2: 0.9211
Fold 3: 0.9123
Fold 4: 0.9123
Fold 5: 0.9558

Mean: 0.9297
Std:  0.0183


In [5]:
# scikit-learn
sklearn_scores = cross_val_score(
    DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),
    X,
    y,
    cv=5
)

print("Fold-by-fold scores:")
for i, score in enumerate(sklearn_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean: {sklearn_scores.mean():.4f}")
print(f"Std:  {sklearn_scores.std():.4f}")

Fold-by-fold scores:
Fold 1: 0.8947
Fold 2: 0.9123
Fold 3: 0.9298
Fold 4: 0.9474
Fold 5: 0.9115

Mean: 0.9191
Std:  0.0180


In [6]:
# comparison
print("From-scratch:")
print(f"Mean ± Std = {scratch_scores.mean():.4f} ± {scratch_scores.std():.4f}")

print("\nScikit-learn:")
print(f"Mean ± Std = {sklearn_scores.mean():.4f} ± {sklearn_scores.std():.4f}")

From-scratch:
Mean ± Std = 0.9297 ± 0.0183

Scikit-learn:
Mean ± Std = 0.9191 ± 0.0180
